## Step 1: Dataset Generation

Synthetic 1D dataset: output follows a square-root curve with noise. A linear model would fail to capture this shape.

In [1]:
import numpy as np
X_input = np.linspace(0, 10, 50).reshape(-1, 1)
y_target = np.sqrt(X_input) + np.random.normal(0, 0.1, X_input.shape)
print("Input shape :", X_input.shape)
print("Target shape:", y_target.shape)
print("X range: %.2f – %.2f" % (X_input.min(), X_input.max()))
print("y range: %.4f – %.4f" % (y_target.min(), y_target.max()))

Input shape : (50, 1)
Target shape: (50, 1)
X range: 0.00 – 10.00
y range: -0.1704 – 3.2023


## Step 2: Activation Functions

ReLU and Sigmoid with their slopes. The slope drives the backward pass.

In [1]:
def relu(z):
    return np.maximum(0, z)
def relu_slope(z):
    return (z > 0).astype(float)
def sigmoid_act(z):
    return 1.0 / (1.0 + np.exp(-z))
def sigmoid_slope(z):
    s = sigmoid_act(z)
    return s * (1.0 - s)
print("relu([-1, 0, 1]):", relu(np.array([-1.,0.,1.])))
print("relu_slope([-1,0,1]):", relu_slope(np.array([-1.,0.,1.])))

relu([-1, 0, 1]): [0. 0. 1.]
relu_slope([-1,0,1]): [0. 0. 1.]


## Step 3: Weight Initialization

Small random weights break symmetry. Biases at zero.

In [1]:
in_dim     = 1
hidden_dim = 4
out_dim    = 1
np.random.seed(0)
W1 = np.random.randn(in_dim, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, out_dim) * 0.1
b2 = np.zeros((1, out_dim))
print("W1 shape:", W1.shape)
print("W2 shape:", W2.shape)
print("b1 shape:", b1.shape)
print("b2 shape:", b2.shape)

W1 shape: (1, 4)
W2 shape: (4, 1)
b1 shape: (1, 4)
b2 shape: (1, 1)


## Step 4: Forward Pass

Data → linear → ReLU → linear output. Output layer has no activation (regression).

In [1]:
def forward_pass(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    H  = relu(Z1)
    out = H @ W2 + b2
    return Z1, H, out

Z1_init, H_init, out_init = forward_pass(X_input, W1, b1, W2, b2)
print("Z1 shape:", Z1_init.shape)
print("H  shape:", H_init.shape)
print("Output shape:", out_init.shape)
print("Initial predictions (first 5):", out_init[:5].flatten().round(4))

Z1 shape: (50, 4)
H  shape: (50, 4)
Output shape: (50, 1)
Initial predictions (first 5): [0.     0.0071 0.0143 0.0214 0.0285]


## Step 5: Backpropagation

Chain rule applied backwards: error → W2 grads → hidden grads through ReLU slope → W1 grads.

In [1]:
def backward_pass(X, y, y_hat, H, Z1, W2):
    N = X.shape[0]
    err      = y_hat - y
    dL_dyhat = 2.0 * err / N
    dL_dW2 = H.T @ dL_dyhat
    dL_db2 = np.sum(dL_dyhat, axis=0, keepdims=True)
    dL_dH  = dL_dyhat @ W2.T
    dL_dZ1 = dL_dH * relu_slope(Z1)
    dL_dW1 = X.T @ dL_dZ1
    dL_db1 = np.sum(dL_dZ1, axis=0, keepdims=True)
    return dL_dW1, dL_db1, dL_dW2, dL_db2
print("backward_pass() defined.")

backward_pass() defined.


## Step 6: Training Loop

Forward → loss → backward → update. Repeats until the model bends to fit the curve.

In [1]:
eta    = 0.01
n_iter = 1000
for it in range(n_iter):
    Z1, H, y_hat = forward_pass(X_input, W1, b1, W2, b2)
    loss = np.mean((y_hat - y_target) ** 2)
    dW1, db1, dW2, db2 = backward_pass(X_input, y_target, y_hat, H, Z1, W2)
    W1 -= eta * dW1
    b1 -= eta * db1
    W2 -= eta * dW2
    b2 -= eta * db2
    if it % 100 == 0:
        print(f"Iteration {it:4d}  |  MSE: {loss:.4f}")

Iteration    0  |  MSE: 4.1690
Iteration  100  |  MSE: 0.0762
Iteration  200  |  MSE: 0.0561
Iteration  300  |  MSE: 0.0503
Iteration  400  |  MSE: 0.0486
Iteration  500  |  MSE: 0.0481
Iteration  600  |  MSE: 0.0480
Iteration  700  |  MSE: 0.0479
Iteration  800  |  MSE: 0.0479
Iteration  900  |  MSE: 0.0479


## Final Check

After training, the output is no longer a straight line. ReLU introduces hinge points that approximate the curve.

In [1]:
_, _, final_out = forward_pass(X_input, W1, b1, W2, b2)
final_loss = np.mean((final_out - y_target)**2)
print(f"Final MSE: {final_loss:.4f}")
print("\nFinal predictions (first 5):")
for i in range(5):
    print(f"  x={X_input[i][0]:.2f}  |  predicted={final_out[i][0]:.4f}  |  true={y_target[i][0]:.4f}")
print("\nThe model now produces a curved output — ReLU neurons fire at different thresholds,")
print("creating a piecewise linear approximation of the sqrt curve.")

Final MSE: 0.0479

Final predictions (first 5):
  x=0.00  |  predicted=0.8058  |  true=-0.1704
  x=0.20  |  predicted=0.8584  |  true=0.3630
  x=0.41  |  predicted=0.9110  |  true=0.6489
  x=0.61  |  predicted=0.9637  |  true=0.6454
  x=0.82  |  predicted=1.0163  |  true=0.9571

The model now produces a curved output — ReLU neurons fire at different thresholds,
creating a piecewise linear approximation of the sqrt curve.
